In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import torch
import torch.nn as nn

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image, preprocess_image

# 1. Cargar modelo entrenado

yolo = YOLO("runs/detect/deteccion_robo/weights/best.pt")

net = yolo.model
net.eval()

# Activar gradientes en el modelo
for p in net.parameters():
    p.requires_grad_(True)

# 2. Wrapper para YOLO

class YOLOWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        output = self.model(x)

        # YOLO puede devolver tuple/list
        if isinstance(output, (tuple, list)):
            output = output[0]

        return output


wrapped_model = YOLOWrapper(net)

# 3. Target para una clase YOLO

class YOLOClassTarget:
    def __init__(self, class_index):
        self.class_index = class_index

    def __call__(self, model_output):
        """
        model_output esperado:
        [batch, canales, predicciones]
        canales = 4 coordenadas + número de clases
        """

        # Si viene con batch: [1, canales, predicciones]
        if model_output.dim() == 3:
            class_scores = model_output[0, 4 + self.class_index, :]

        # Si viene sin batch: [canales, predicciones]
        elif model_output.dim() == 2:
            class_scores = model_output[4 + self.class_index, :]

        else:
            raise ValueError(f"Forma de salida no esperada: {model_output.shape}")

        return class_scores.max()

# 4. Clase a explicar

# clases:
# 0 = normal
# 1 = robo
class_index = 1  # robo

targets = [YOLOClassTarget(class_index)]

# 5. Capa objetivo

target_layers = [net.model[-2]]

# Si sale mal:
# target_layers = [net.model[-3]]
# target_layers = [net.model[-4]]

# 6. Leer imagen

image_path = "robo3.jpeg"

bgr_img = cv2.imread(image_path)

if bgr_img is None:
    print("No se pudo leer la imagen:", image_path)
    exit()

rgb_img = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
rgb_img = cv2.resize(rgb_img, (640, 640))

rgb_float = np.float32(rgb_img) / 255.0

# 7. Convertir imagen a tensor

input_tensor = preprocess_image(
    rgb_float,
    mean=[0, 0, 0],
    std=[1, 1, 1]
)

# Activar gradiente en la entrada
input_tensor.requires_grad_(True)

# 8. Generar Grad-CAM

cam = GradCAM(
    model=wrapped_model,
    target_layers=target_layers
)

grayscale_cam = cam(
    input_tensor=input_tensor,
    targets=targets
)[0]

# 9. Superponer mapa de calor

cam_image = show_cam_on_image(
    rgb_float,
    grayscale_cam,
    use_rgb=True
)

cam_image_bgr = cv2.cvtColor(cam_image, cv2.COLOR_RGB2BGR)

# 10. Mostrar y guardar

cv2.imwrite("gradcam_robo.jpg", cam_image_bgr)

cv2.imshow("Grad-CAM YOLO - Robo", cam_image_bgr)
cv2.waitKey(0)
cv2.destroyAllWindows()